# Buddy RVC GPU Lab — Android Free

This notebook is the heavy-compute escape hatch for Little Red's Big Studio. Run it from Android in Google Colab or import the notebook into Kaggle. It uses the official Applio project for RVC training/inference. Your voice sample must be yours or authorized.

**Goal:** train a persistent Buddy RVC voice model once, then reuse the `.pth` + `.index` instead of repeatedly spending ZeroGPU quota on zero-shot cloning.

In [ ]:
import os, subprocess, sys, pathlib, shutil
print('Python:', sys.version)
gpu = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(gpu.stdout if gpu.returncode == 0 else 'No NVIDIA GPU detected. Enable a GPU runtime before training.')

In [ ]:
%cd /content
if not os.path.exists('/content/Applio'):
    !git clone --depth 1 https://github.com/IAHispano/Applio.git
%cd /content/Applio
!bash run-install.sh

## Put your authorized training audio in Drive

Use a clean dataset of your own voice. For a strong RVC model, use multiple clean recordings rather than a single short clip. Keep background music, reverb and other speakers out of the training set.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATASET = '/content/drive/MyDrive/Little Reds Big Studio/BuddyVoice/dataset'
MODEL = 'Buddy'
os.makedirs(DATASET, exist_ok=True)
print('Dataset folder:', DATASET)
print('Files:', len(list(pathlib.Path(DATASET).glob('*'))))

## Launch Applio

The Applio interface handles preprocessing, feature extraction, training, index generation and inference. On Colab, launch it with its Colab mode and open the generated public Gradio URL.

Recommended starting point: **RVC v2, RMVPE, 32 kHz or 40 kHz, batch size appropriate to the assigned GPU, and enough epochs to converge without overtraining.**

In [ ]:
%cd /content/Applio
!python app.py --colab

## After training

Download the generated Buddy `.pth` model and `.index` file from Applio. Keep both together. Little Red's Big Studio can then use the persistent RVC model for TTS-to-RVC and singing voice conversion without repeatedly asking a public ZeroGPU Space to clone the same voice.

In [ ]:
from pathlib import Path
weights = list(Path('/content/Applio').rglob('*.pth'))
indexes = list(Path('/content/Applio').rglob('*.index'))
print('PTH models:', *[str(x) for x in weights[-10:]], sep='\n')
print('INDEX files:', *[str(x) for x in indexes[-10:]], sep='\n')